In [1]:
import pandas as pd
import epi_utils as eu

In [2]:
from tqdm.auto import tqdm
tqdm.pandas()

In [3]:
from sqlalchemy import create_engine
import pymysql

In [34]:
username = "root"
password = ""
port = 3306
database = "hg38"

In [35]:
engine = create_engine('mysql+pymysql://%s@localhost:%i/%s' %(username, port, database))

# Preparing NCBI RefSeq hg38 data

In [37]:
sql = "SELECT * FROM ncbirefseq"
ncbi_df = pd.read_sql_query(sql, engine)

display(ncbi_df.head())
display(ncbi_df.shape)

,bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,585,NR_046018.2,chr1,+,11873,14409,14409,14409,3,"b'11873,12612,13220,'","b'12227,12721,14409,'",0,DDX11L1,none,none,"b'-1,-1,-1,'"
1,585,NR_024540.1,chr1,-,14361,29370,29370,29370,11,"b'14361,14969,15795,16606,16857,17232,17605,17...","b'14829,15038,15947,16765,17055,17368,17742,18...",0,WASH7P,none,none,"b'-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,'"
2,585,NR_106918.1,chr1,-,17368,17436,17436,17436,1,"b'17368,'","b'17436,'",0,MIR6859-1,none,none,"b'-1,'"
3,585,XR_007065314.1,chr1,+,29773,35418,35418,35418,3,"b'29773,30975,34167,'","b'30667,31093,35418,'",0,MIR1302-2HG,none,none,"b'-1,-1,-1,'"
4,585,NR_036051.1,chr1,+,30365,30503,30503,30503,1,"b'30365,'","b'30503,'",0,MIR1302-2,none,none,"b'-1,'"


(196097, 16)

In [ ]:
ncbi_df.loc[:, "tss"] = ncbi_df.progress_apply(lambda row: eu.get_tss_ncbi(row), axis=1)

In [ ]:
ncbi_df.columns

In [ ]:
ncbi_df = ncbi_df[['chrom', 'txStart', 'txEnd', 'name', 'score', 'strand', 'bin', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss']]

In [ ]:
ncbi_df.head()

In [ ]:
ncbi_df.to_csv("dataset/ncbi_refseq_hg38.bed", header=False, index=False, sep="\t")

# Reading bedfile intersection

In [38]:
columns = [['chrom', 'chromStart', 'chromEnd', 'test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2', 'status',
       'value_1', 'value_2', 'log2(fold_change)', 'test_stat', 'p_value',
       'q_value', 'significant', 'chrom', 'txStart', 'txEnd', 'name', 'score', 'strand', 'bin', 'cdsStart',
       'cdsEnd', 'exonCount', 'exonStarts', 'exonEnds', 'name2',
       'cdsStartStat', 'cdsEndStat', 'exonFrames', 'tss']]

In [39]:
overlap_df = pd.read_csv("dataset/hepg2_ncbirefseq_hg38_overlap80.bed", sep="\t", header=None)
overlap_df.columns = columns[:len(overlap_df.columns)]

In [40]:
display(overlap_df.head())
display(overlap_df.shape)

,chrom,chromStart,chromEnd,test_id,gene_id,gene,locus,sample_1,sample_2,status,...,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,name2,cdsStartStat,cdsEndStat,exonFrames,tss
0,chr1,69090,70008,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,...,65564,70008,3,"b'65418,65519,69036,'","b'65433,65573,71585,'",OR4F5,cmpl,cmpl,"b'-1,0,0,'",65418
1,chr1,367658,368597,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,...,365554,382235,5,"b'365133,373143,379768,380896,382049,'","b'365692,373323,379870,381688,382235,'",LOC112268260,cmpl,cmpl,"b'0,0,0,0,0,'",382235
2,chr1,840263,843900,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,...,843604,843604,3,"b'827590,829002,841199,'","b'827775,829104,843604,'",LINC01128,none,none,"b'-1,-1,-1,'",827590
3,chr1,840263,843900,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,...,859446,859446,4,"b'827590,829002,852670,853390,'","b'827775,829104,852766,859446,'",LINC01128,none,none,"b'-1,-1,-1,-1,'",827590
4,chr1,840263,843900,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,...,859446,859446,4,"b'827590,829002,851926,853390,'","b'827775,829104,852110,859446,'",LINC01128,none,none,"b'-1,-1,-1,-1,'",827590


(100066, 34)

In [41]:
overlap_df.to_csv("dataset/hepg2_ncbirefseq_hg38_overlap80.csv", header=True, index=False)

In [48]:
gene_id_df = overlap_df['gene_id']
unique_rows = gene_id_df.drop_duplicates()
display(unique_rows)

,gene_id
0,XLOC_000001
1,XLOC_000003
2,XLOC_000005
8,XLOC_000008
9,XLOC_000016
...,...
99481,XLOC_030026
99594,XLOC_030027
99707,XLOC_030028
99820,XLOC_030029


In [46]:
gene_df = overlap_df['gene']
display(gene_id_df.head())

,gene
0,OR4F5
1,OR4F29
2,-
3,-
4,-


In [47]:
unique_rows = gene_df.drop_duplicates()
display(unique_rows)

,gene
0,OR4F5
1,OR4F29
2,-
8,ISG15
36,GLTPD1
...,...
99358,TTTY23
99472,"NCRNA00230A,NCRNA00230B"
99473,ASMTL
99479,SRY
